In [25]:
import os
import zipfile
import tempfile
import geopandas as gpd
import pandas as pd
import fiona

In [26]:
USER = '157412'
base_dir = f'C:\\Users\\{USER}\\OneDrive - UTS (1)\\ISF Projects - 23204_Quadrature_Mineral Availability for Renewables PEMMS\\3. Resources\\Graphite Production\\Sciencebase data'

commodity_nomenclature = ['COMMOD', 'MINERAL', 'MATER']

def inspect_gdb(gdb_path):
    """Prints all layers and the columns of the first layer found."""
    layers = gpd.list_layers(gdb_path)
    print("--- Layers in GDB ---")
    print(layers[['name', 'geometry_type']])
    return layers['name'].tolist()

def load_graphite(gdb_path, layer_name):
    """Loads a specific layer and filters for graphite."""
    gdf = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio")
    # Identify the commodity column (USGS usually uses 'COMMODITY', 'COMMOD_1', or 'MINERAL')
    comm_col = [c for c in gdf.columns if any(x in c.upper() for x in commodity_nomenclature)]
    
    if not comm_col:
        print(f"Warning: No commodity column found in {layer_name}. Columns: {gdf.columns.tolist()[:5]}...")
        return gdf # Return all if we can't filter automatically
    
    # Filter (case-insensitive)
    col = comm_col[0]
    mask = gdf[col].astype(str).str.contains('graphite', case=False, na=False)
    filtered = gdf[mask].copy()
    print(f"Found {len(filtered)} graphite records in {layer_name} using column '{col}'")
    return filtered

def get_alias_map(gdb_path, layer_name):
    """
    Extracts alias mapping. Handles the zip:// prefix required by fiona for compressed GDBs.
    """
    # Fiona requires a specific URI format for zipped files
    if '.gdb.zip' in gdb_path:
        # Format: zip://C:/path/to/data.gdb.zip!Layer_Name
        fiona_path = f"zip://{gdb_path}"
    else:
        fiona_path = gdb_path

    try:
        with fiona.open(fiona_path, layer=layer_name) as layer:
            # Attempt to get aliases; if not found, use field names
            aliases = getattr(layer, 'field_aliases', {})
            if not aliases:
                # Fallback: some versions store it in the 'meta' or 'schema'
                return {name: name for name in layer.schema['properties'].keys()}
            return aliases
    except Exception as e:
        print(f"      Warning: Fiona could not read aliases for {layer_name}: {e}")
        return {}

def load_usgs_flexible(gdb_path, region_name):
    target_keywords = ['Facilities', 'Deposits', 'Exploration', 'Development']
    all_graphite = []
    
    try:
        layers = gpd.list_layers(gdb_path)['name'].tolist()
        mining_layers = [l for l in layers if any(k in l for k in target_keywords)]
        
        for layer in mining_layers:
            print(f"Inspecting {region_name} -> {layer}...")
            
            # 1. Load the raw data
            gdf = gpd.read_file(gdb_path, layer=layer, engine="pyogrio")
            
            # 2. Get and Apply Aliases immediately
            alias_map = get_alias_map(gdb_path, layer)
            if alias_map:
                # Ensure we only try to rename columns that actually exist in the GDF
                current_aliases = {k: v for k, v in alias_map.items() if k in gdf.columns}
                gdf = gdf.rename(columns=current_aliases)
                print(f"  --> Applied {len(current_aliases)} aliases.")

            # 3. Identify all text columns (which now have alias names)
            text_cols = gdf.select_dtypes(include=['object']).columns
            
            # 4. Filter for 'graphite'
            mask = gdf[text_cols].astype(str).apply(
                lambda x: x.str.contains('graphite', case=False, na=False)
            ).any(axis=1)
            
            hits = gdf[mask].copy()
            
            if not hits.empty:
                hits['Source_Region'] = region_name
                hits['Source_Layer'] = layer
                all_graphite.append(hits)
                print(f"  --> Found {len(hits)} graphite records.")
            else:
                print(f"  (No graphite in this layer)")
                
    except Exception as e:
        print(f"Error processing {region_name}: {e}")
        
    if all_graphite:
        # pd.concat will align columns by name. 
        # 'Commodity 1' from Deposits will align with 'Commodity 1' from Exploration.
        return pd.concat(all_graphite, ignore_index=True)
    return None

# 2. Loading GDBs per region

## 2.1. Africa

In [27]:
# Path to Africa GDB
africa_path = os.path.join(base_dir, "Compilation of GIS Africa", "Africa_GIS.gdb.zip")
# Inspect
layers = inspect_gdb(africa_path)

--- Layers in GDB ---
                                name      geometry_type
0             AFR_Mineral_Facilities            Point Z
1               AFR_Mineral_Deposits            Point Z
2            AFR_Mineral_Exploration            Point Z
3         AFR_Mineral_Resources_Coal       MultiPolygon
4       AFR_Mineral_Resources_Copper       MultiPolygon
5        AFR_Mineral_Resources_Gabon       MultiPolygon
6   AFR_Mineral_Resources_Mauritania       MultiPolygon
7          AFR_Mineral_Resources_PGE       MultiPolygon
8       AFR_Mineral_Resources_Potash       MultiPolygon
9             AFR_Infra_OG_Pipelines    MultiLineString
10          AFR_Infra_Power_Stations            Point Z
11       AFR_OG_Provinces_Continuous       MultiPolygon
12     AFR_OG_Provinces_Conventional       MultiPolygon
13      AFR_OG_Resources_Recoverable       MultiPolygon
14         AFR_Infra_Transport_Ports            Point Z
15          AFR_Infra_Transport_Rail    MultiLineString
16              AFR_Politi

c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\core.py:130: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\core.py:130: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))


In [ ]:
AFRICA_ALIASES = {
    # --- AFR_Mineral_Facilities ---
    # 2,408 facilities in 1,866 locations from USGS 2018 Minerals Yearbook Vol. III
    # Data reflects state of minerals industries as of 2018 (Zambia: 2017)
    # LocConfid: 'A' (approximate), 'E' (exact)
    # FeatureTyp: 'Oil and Gas Fields', 'Mines and Quarries', 'Mineral Processing Plants',
    #   'Oil and Gas Refineries and (or) Petrochemical Complexes', 'Brine'
    # DsgAttr01 (Commodity Type): 'Fuel', 'Industrial', 'Metal', 'Fertilizer', 'Gemstone'
    # DsgAttr04 (Multiple Commodities): 'Y'/'N'
    # DsgAttr05 (Multiple Products): 'Y'/'N'
    # DsgAttr06 (Year): 2017–2018
    # DsgAttr07 (Annual Production Capacity): -999 = unknown; range 1–22,600,000
    # DsgAttr08 (Capacity Units): 'NA', 'Kilograms', 'Metric tons', 'Thousand metric tons',
    #   'Million 42-gallon barrels', 'Million cubic meters', 'Cubic meters',
    #   '42-gallon barrels per day', 'Thousand 42-gallon barrels',
    #   'Thousand 42-gallon barrels per day', 'Barrels of liquids per day',
    #   'Thousand bricks', 'Thousand carats'
    # DsgAttr09 (Capacity Notes): null=none, 'c'=combined, 'e'=estimated, 'g'=gross weight
    #   NOTE: lowercase codes (unlike China which uses uppercase C/E/G)
    # DsgAttr10 (Shared Capacity): lists FeatureUIDs sharing the same capacity
    # LocOpStat: 'Active', 'Assumed Active', 'C&M' (care & maintenance),
    #   'Closed', 'Construction', 'Development', 'Suspended'
    # Lat range: -35.31–37.26, Lon range: -23.73–57.61
    'AFR_Mineral_Facilities': {
        'FeatureUID': 'USGS Facility ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'LocConfid': 'Location Accuracy',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Description',
        'FeatureTyp': 'Facility Type',
        'FeatureNam': 'Facility Name',
        'Label1': 'Label',
        'DsgAttr01': 'Commodity Type',
        'DsgAttr02': 'Commodity',
        'DsgAttr03': 'Commodity Product',
        'DsgAttr04': 'Multiple Commodities',
        'DsgAttr05': 'Multiple Products',
        'DsgAttr06': 'Year',
        'DsgAttr07': 'Annual Production Capacity',
        'DsgAttr08': 'Capacity Units',
        'DsgAttr09': 'Capacity Notes',
        'DsgAttr10': 'Shared Capacity',
        'LocOpStat': 'Facility Status',
        'MemoOther': 'Facility Comments',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'InfSource1': 'Mineral Facility Data Source',
        'LocSource1': 'GIS Data Source'
    },

    # --- AFR_Mineral_Deposits ---
    # Selected deposits from USGS PP 1802 and OFR 2005-1294
    # 8 commodity fields (DsgAttr01–08), plus Deposit Type and MRData ID
    'AFR_Mineral_Deposits': {
        'FeatureUID': 'Deposit ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Deposit Name',
        'NameVar': 'Alternate Name(s)',
        'Label1': 'Label',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Commodity 8',
        'DsgAttr09': 'Deposit Type',
        'DsgAttr10': 'USGS MRData ID',
        'InfSource1': 'Data Source'
    },

    # --- AFR_Mineral_Exploration ---
    # Exploration sites from USGS World Minerals Exploration Database (WMED)
    # LocOpStat: variable/non-standardized text (not enumerated like China)
    # DsgAttr09 (Owner Type): variable text; empty cell = not available
    # DsgAttr10 (Owner Share): percentage equity share
    # DsgAttr11 (Project Active): year last verified active, range 1995–2018
    # DsgAttr12 (USGS WMED ID): internal WMED identifier
    # Lat range: -33.10–37.05, Lon range: -16.79–49.16
    'AFR_Mineral_Exploration': {
        'FeatureUID': 'Exploration ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name(s)',
        'Label1': 'Label',
        'MemoLoc': 'Location Notes',
        'LocOpStat': 'Exploration Status',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Commodity 8',
        'OwnerName': 'Owner Name',
        'DsgAttr09': 'Owner Type',
        'DsgAttr10': 'Owner Share',
        'DsgAttr11': 'Project Active',
        'DsgAttr12': 'USGS WMED ID',
        'InfSource1': 'Data Source'
    }
}

In [29]:
def load_africa_data_and_aliases(gdb_path):
    all_graphite = []
    
    # We only iterate through the layers defined in our manual map
    for layer_name, mapping in AFRICA_ALIASES.items():
        try:
            print(f"Processing {layer_name}...")
            # Load layer with pyogrio for speed
            gdf = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio")
            
            # Rename using our verified alias dictionary
            gdf = gdf.rename(columns=mapping)
            
            # Search for 'graphite' across all columns to ensure we don't miss secondary commodities
            mask = gdf.astype(str).apply(
                lambda x: x.str.contains('graphite', case=False, na=False)
            ).any(axis=1)
            
            hits = gdf[mask].copy()
            
            if not hits.empty:
                hits['Source_Layer'] = layer_name
                all_graphite.append(hits)
                print(f"  --> Success: Found {len(hits)} graphite records.")
            else:
                print(f"  --> No graphite found.")
                
        except Exception as e:
            print(f"  --> Error loading {layer_name}: {e}")

    if all_graphite:
        # Aligning all hits into a single Master Africa GeoDataFrame
        master_africa = pd.concat(all_graphite, ignore_index=True)
        return master_africa
    return None

In [30]:
africa_master = load_africa_data_and_aliases(africa_path)

if africa_master is not None:
    display(africa_master.head(10))

c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(


Processing AFR_Mineral_Facilities...
  --> Success: Found 8 graphite records.
Processing AFR_Mineral_Deposits...


c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(


  --> Success: Found 22 graphite records.
Processing AFR_Mineral_Exploration...
  --> Success: Found 41 graphite records.


c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_read(


,USGS Facility ID,Label,Country (Short Form),Facility Name,Facility Type,Commodity Type,Commodity,Commodity Product,Multiple Commodities,Multiple Products,...,USGS MRData ID,Exploration ID,Site Name,Exploration Status,Location Notes,Owner Name,USGS WMED ID,Project Active,Owner Type,Owner Share
0,MDG007,MDG7,Madagascar,Gallois Mine,Mines and Quarries,Industrial,Graphite,<null>,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MDG008,MDG8,Madagascar,Graphmada Mine,Mines and Quarries,Industrial,Graphite,<null>,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MDG023,MDG23,Madagascar,Mine in Atsinanana Region,Mines and Quarries,Industrial,Graphite,<null>,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MDG044,MDG45,Madagascar,Sahamamy-Sahasoa Mine,Mines and Quarries,Industrial,Graphite,<null>,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MOZ001,MOZ1,Mozambique,Ancuabe Graphite Mine,Mines and Quarries,Industrial,Graphite,Graphite,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,MOZ002,MOZ2,Mozambique,Balama Graphite Operation,Mines and Quarries,Industrial,Graphite,Graphite,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NAM017,NAM17,Namibia,Okanjande Mine,Mines and Quarries,Industrial,Graphite,Graphite,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ZWE043,ZWE43,Zimbabwe,Lynx Graphite Mine,Mines and Quarries,Industrial,Graphite,Graphite,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,KEd7,Kenya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,KEd8,Kenya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1115.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2.2. China

In [31]:
china_path = os.path.join(base_dir, "Compilation of GIS China", "CHN_GIS.gdb.zip")
# Inspect
layers = inspect_gdb(china_path)

--- Layers in GDB ---
                                   name    geometry_type
0        CHN_Mineral_Resources_Antimony     MultiPolygon
1            CHN_Mineral_Resources_Coal     MultiPolygon
2          CHN_Mineral_Resources_Copper     MultiPolygon
3       CHN_Mineral_Resources_Phosphate     MultiPolygon
4          CHN_Mineral_Resources_Potash     MultiPolygon
5                  CHN_Mineral_Deposits            Point
6            CHN_Infra_OG_LNG_Terminals            Point
7                        CHN_Infra_Dams            Point
8          CHN_Infra_Power_Transmission  MultiLineString
9             CHN_Infra_Transport_Ports            Point
10          CHN_OG_Provinces_Continuous     MultiPolygon
11  CHN_OG_Province_Groups_Conventional     MultiPolygon
12         CHN_OG_Resources_Recoverable     MultiPolygon
13        CHN_OG_Provinces_Conventional     MultiPolygon
14              CHN_Mineral_Exploration            Point
15               CHN_Mineral_Facilities            Point
16       

In [ ]:
CHINA_ALIASES = {
    # --- CHN_Mineral_Facilities ---
    # 2,424 mineral production/processing facilities from USGS 2019 Minerals Yearbook Vol. III
    # Data reflects state of minerals industries as of 2019
    # DsgAttr03 (Multiple Commodities): 'Y'/'N' - multiple commodities at one facility
    # DsgAttr04 (Multiple Products): 'Y'/'N' - multiple products at one facility
    # DsgAttr06 (Annual Production Capacity): -999 = unknown/unavailable; range 1~130,000
    # DsgAttr07 (Capacity Units): 
        #   'NA', 'kg' (kilograms), 'mt' (metric tons), 'MMbbl' (Million 42-gal barrels),
        #   'mcm' (Million cubic meters), 'million mt', 'kmt' (thousand mt)
    # DsgAttr08 (Capacity Notes): 
        # null=none, 'C'=combined facilities, 'E'=estimated, 'G'=gross weight
    # DsgAttr09 (Shared Capacity): lists FeatureUIDs of other facilities sharing the same capacity
    'CHN_Mineral_Facilities': {
        'FeatureUID': 'USGS Facility ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'LocConfid': 'Location Confidence',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Notes',
        'FeatureTyp': 'Feature Type',
        'FeatureNam': 'Facility Name',
        'Label': 'Label',
        'DsgAttr01': 'Commodity',
        'DsgAttr02': 'Commodity Product',
        'DsgAttr03': 'Multiple Commodities',
        'DsgAttr04': 'Multiple Products',
        'DsgAttr05': 'Product Notes',
        'DsgAttr06': 'Annual Production Capacity',
        'DsgAttr07': 'Capacity Units',
        'DsgAttr08': 'Capacity Notes',
        'DsgAttr09': 'Shared Capacity',
        'LocOpStat': 'Operating Status',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'OwnerName5': 'Equity Owner 5',
        'InfSource1': 'Mineral Facility Data Source',
        'LocSource1': 'GIS Data Source'
        },

    # --- CHN_Mineral_Deposits ---
    # Selected deposits from USGS OFR 2005-1294 and Professional Paper 1802
    # Only 4 commodity fields (DsgAttr01–04), unlike Exploration which has 7
    # DsgAttr05 is Deposit Type (not Commodity 5)
    # Note: COUNTRY, LATITUDE, LONGITUDE are UPPERCASE (unlike other layers)
    # Lat range: 18.7166–53, Lon range: 81.2499–132.4833
    # Label1 is just FeatureUID repeated for map labeling
    'CHN_Mineral_Deposits': {
        'FeatureUID': 'Deposit ID',
        'FeatureNam': 'Deposit Name',
        'NameVar': 'Alternate Name(s)',
        'COUNTRY': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'LATITUDE': 'DD Latitude',
        'LONGITUDE': 'DD Longitude',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Deposit Type',
        'Label1': 'Label',
        'InfSource1': 'Data Source',
    },

    # --- CHN_Mineral_Exploration ---
    # 386 exploration sites from USGS World Minerals Exploration Database (WMED)
    # LocOpStat valid values: 'Exploration', 'Feasibility Study', 'Inactive', 'Pre-production'
    # LocConf valid values: 'A' (approximated), 'E' (exact)
    # DsgAttr09 (Source Year) range: 1994–2022
    # Lat range: 21.48–53.0082, Lon range: 74.26–132.31
    # Note: admin field is AMD1 (not ADM1 like other layers)
    # Low-accuracy locations use ADM1 centroid coordinates
    'CHN_Mineral_Exploration': {
        'FeatureUID': 'Exploration ID',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Location Description',
        'Country': 'Country (Short Form)',
        # Note THAT THERE AS A TYPO IN THE ORIGINAL DATA. Hence the "AMD" instead of "ADM"
        'AMD1': 'AMD1 Name',
        'LocOpStat': 'Exploration Status',
        'MemoLoc': 'Location Notes',
        'LocConf': 'Location Confidence',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'OwnerName': 'Owner Name',
        'DsgAttr09': 'Source Year',
        'InfSource1': 'Location Source 1',
        'InfSource2': 'Location Source 2',
        'InfSource3': 'Operation Source 1',
        'InfSource4': 'Operation Source 2',
        'InfSource5': 'Company Source 1',
        'InfSource6': 'Company Source 2',
        'InfSource7': 'Commodity Source 1',
        'InfSource8': 'Commodity Source 2',
    }
}

In [33]:
def load_china_data_and_aliases(gdb_path):
    all_graphite = []
    
    for layer_name, mapping in CHINA_ALIASES.items():
        try:
            print(f"Processing {layer_name}...")
            gdf = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio")
            gdf = gdf.rename(columns=mapping)
            
            # Case-insensitive search across all renamed text columns
            mask = gdf.astype(str).apply(
                lambda x: x.str.contains('graphite', case=False, na=False)
            ).any(axis=1)
            
            hits = gdf[mask].copy()
            
            if not hits.empty:
                hits['Source_Layer'] = layer_name
                hits['Region'] = 'China'
                all_graphite.append(hits)
                print(f"  --> Success: Found {len(hits)} graphite records.")
            else:
                print(f"  --> No graphite found.")
                
        except Exception as e:
            print(f"  --> Error loading {layer_name}: {e}")

    if all_graphite:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            return pd.concat(all_graphite, ignore_index=True)
    return None

china_master = load_china_data_and_aliases(china_path)

if china_master is not None:
    display(china_master.head(10))

Processing CHN_Mineral_Facilities...
  --> Success: Found 5 graphite records.
Processing CHN_Mineral_Deposits...
  --> Success: Found 11 graphite records.
Processing CHN_Mineral_Exploration...
  --> Success: Found 6 graphite records.


,USGS Facility ID,Label,Country (Short Form),Facility Name,Feature Type,Commodity,Commodity Product,Multiple Commodities,Multiple Products,Product Notes,...,LocConf,Owner Name,Owner Type,Location Source 2,InfSource3,InfSource4,InfSource5,InfSource6,InfSource7,InfSource8
0,CHN0500,CHN500,China,Liumao Mine,Mines and Quarries,Graphite,None,N,N,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CHN0501,CHN501,China,Mine in Jixi,Mines and Quarries,Graphite,None,N,N,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CHN1095,CHN1095,China,Mine in Xinghe,Mines and Quarries,Graphite,None,N,N,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CHN1434,CHN1434,China,Plant in Qingdao,Mineral Processing Plants,Graphite,None,N,N,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CHN1467,CHN1467,China,Qingdao Mine,Mines and Quarries,Graphite,None,N,N,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,CHNd24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,CHNd25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,CHNd210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,CHNd211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,CHNd212,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2.3. Indopacific

In [34]:
indopac_path = os.path.join(base_dir, "Compilation of GIS Indopacific", "INDOPAC_GIS.gdb.zip")

# Inspect
layers = inspect_gdb(indopac_path)

--- Layers in GDB ---
                                name geometry_type
0        INDOPAC_Mineral_Development         Point
1        INDOPAC_Mineral_Exploration         Point
2         INDOPAC_Mineral_Facilities         Point
3     INDOPAC_Mineral_Resources_Coal  MultiPolygon
4   INDOPAC_Mineral_Resources_Copper  MultiPolygon
5    INDOPAC_OG_Provinces_Continuous  MultiPolygon
6  INDOPAC_OG_Provinces_Conventional  MultiPolygon
7   INDOPAC_OG_Resources_Recoverable  MultiPolygon


In [ ]:
INDOPAC_ALIASES = {
    # --- INDOPAC_Mineral_Facilities ---
    # Fuel and non-fuel mineral production/processing facilities in Indo-Pacific nations
    # FeatureTyp: 'Brine', 'Mineral Processing Plants', 'Mines and Quarries',
    #   'Oil and Gas Fields', 'Oil and Gas Refineries and (or) Petrochemical Complexes'
    # NOTE: No 'Commodity Type' field (unlike Africa) — DsgAttr01 is Commodity directly
    # DsgAttr04 (Multiple Commodities): 'Y'/'N'
    # DsgAttr05 (Multiple Products): 'Y'/'N'
    # DsgAttr06 (Annual Production Capacity): -999 = unknown; range 1–110,000
    # DsgAttr07 (Capacity Units): 'NA', 'kbbl', 'kg', 'kmt', 'mcm',
    #   'million mt', 'MMbbl', 'mt', 'Thousand carats'
    # DsgAttr08 (Capacity Notes): Null=none, 'C'=combined, 'E'=estimated, 'G'=gross weight
    # DsgAttr09 (Shared Capacity): lists FeatureUIDs sharing the same capacity
    # LocOpStat: 'Assumed Active', 'C&M' (care & maintenance), 'Suspended'
    # LocConf: 'A' (approximate), 'E' (exact)
    # Lat range: -46.59–50.32, Lon range: 79.82–178.72
    # NOTE: uses LocConf (not LocConfid), Label (not Label1)
    # Has 6 equity owners (OwnerName1–6), more than other regions
    'INDOPAC_Mineral_Facilities': {
        'FeatureUID': 'USGS Facility ID',
        'Country': 'Country (Short Form)',
        'FeatureNam': 'Facility Name',
        'FeatureTyp': 'Feature Type',
        'DsgAttr01': 'Commodity',
        'DsgAttr02': 'Commodity Product',
        'DsgAttr03': 'Product Notes',
        'DsgAttr04': 'Multiple Commodities',
        'DsgAttr05': 'Multiple Products',
        'DsgAttr06': 'Annual Production Capacity',
        'DsgAttr07': 'Capacity Units',
        'DsgAttr08': 'Capacity Notes',
        'DsgAttr09': 'Shared Capacity',
        'LocOpStat': 'Facility Status',
        'DsgAttr10': 'Location Description',
        'ADM1': 'ADM1 Name',
        'LocConf': 'Location Accuracy',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'OwnerName5': 'Equity Owner 5',
        'OwnerName6': 'Equity Owner 6',
        'Label': 'Label',
        'InfSource1': 'Location Source 1',
        'InfSource2': 'Location Source 2',
        'InfSource3': 'Capacity Source 1',
        'InfSource4': 'Capacity Source 2',
        'InfSource5': 'Company Source 1',
        'InfSource6': 'Company Source 2',
        'InfSource7': 'Other Source',
        'MemoSource': 'Source notes',
    },

    # --- INDOPAC_Mineral_Development ---
    # 285 development sites across Indo-Pacific nations (Bangladesh, Bhutan, Brunei,
    #   Burma, Fiji, Malaysia, Mongolia, New Caledonia, New Zealand, Papua New Guinea,
    #   Philippines, Singapore, Solomon Islands, South Korea, Sri Lanka, Taiwan, Vietnam)
    # NOTE: Only 6 commodity fields (DsgAttr01–06), then DsgAttr07 = Commodity Product
    # DsgAttr08 = Location Description (NOT Commodity 8)
    # FeatureTyp: 'Mineral Processing Plants', 'Mines and Quarries',
    #   'Oil and Gas Fields', 'Oil and Gas Refineries and (or) Petrochemical Complexes'
    # LocOpStat: 'Construction', 'Feasibility', 'Inactive', 'Planned Construction', 'Pre-Production'
    # LocConf: 'A' (approximate), 'E' (exact)
    # DsgAttr09 (Source Year): range 2014–2023
    # Lat range: -43.50–49.56, Lon range: 80.50–179.27
    # Has OperateNam + OwnerName1–5 (not single OwnerName like Africa Exploration)
    'INDOPAC_Mineral_Development': {
        'FeatureUID': 'Development ID',
        'Country': 'Country (Short Form)',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name(s)',
        'FeatureTyp': 'Feature Type',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity Product',
        'LocOpStat': 'Development Status',
        'DsgAttr08': 'Location Description',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Notes',
        'LocConf': 'Location Accuracy',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'OwnerName5': 'Equity Owner 5',
        'Label': 'Label',
        'DsgAttr09': 'Source Year',
        'InfSource1': 'Location Source 1',
        'InfSource2': 'Location Source 2',
        'InfSource3': 'Commodity Source 1',
        'InfSource4': 'Commodity Source 2',
        'InfSource5': 'Operating Stage Source 1',
        'InfSource6': 'Operating Stage Source 2',
        'InfSource7': 'Company Source 1',
        'InfSource8': 'Company Source 2',
        'InfSource9': 'Other Source',
        'MemoSource': 'Source notes',
    },

    # --- INDOPAC_Mineral_Exploration ---
    # Exploration sites from USGS WMED across Indo-Pacific nations
    # NOTE: Only 6 commodity fields (DsgAttr01–06), then DsgAttr07 = Location Description
    # LocOpStat: 'Exploration', 'Inactive', 'Preliminary Study'
    # LocConf: 'A' (approximate), 'E' (exact)
    # DsgAttr09 (Source Year): range 1998–2023
    # Lat range: -46.55–50.55, Lon range: 78.96–179.96
    # Has OperateNam + OwnerName1–4 (4 owners, not 5 like Development)
    # No DsgAttr08 — Location Description is DsgAttr07 here (vs DsgAttr08 in Development)
    'INDOPAC_Mineral_Exploration': {
        'FeatureUID': 'Exploration ID',
        'Country': 'Country (Short Form)',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name(s)',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'LocOpStat': 'Exploration Status',
        'DsgAttr07': 'Location Description',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Notes',
        'LocConf': 'Location Accuracy',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'DsgAttr09': 'Source Year',
        'InfSource1': 'Location Source 1',
        'InfSource2': 'Location Source 2',
        'InfSource3': 'Commodity Source 1',
        'InfSource4': 'Commodity Source 2',
        'InfSource5': 'Operating Stage Source 1',
        'InfSource6': 'Operating Stage Source 2',
        'InfSource7': 'Company Source 1',
        'InfSource8': 'Company Source 2',
        'InfSource9': 'Other Source',
        'Label': 'Label',
    },
}

In [36]:

def load_indopac_data_and_aliases(gdb_path):
    all_graphite = []
    
    for layer_name, mapping in INDOPAC_ALIASES.items():
        try:
            print(f"Processing {layer_name}...")
            gdf = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio")
            gdf = gdf.rename(columns=mapping)
            
            mask = gdf.astype(str).apply(
                lambda x: x.str.contains('graphite', case=False, na=False)
            ).any(axis=1)
            
            hits = gdf[mask].copy()
            
            if not hits.empty:
                hits['Source_Layer'] = layer_name
                hits['Region'] = 'Indopacific'
                all_graphite.append(hits)
                print(f"  --> Success: Found {len(hits)} graphite records.")
            else:
                print(f"  --> No graphite found.")
                
        except Exception as e:
            print(f"  --> Error loading {layer_name}: {e}")

    if all_graphite:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            return pd.concat(all_graphite, ignore_index=True)
    return None

In [37]:
indopac_master = load_indopac_data_and_aliases(indopac_path)

if indopac_master is not None:
    display(indopac_master.head(10))

Processing INDOPAC_Mineral_Facilities...
  --> Success: Found 6 graphite records.
Processing INDOPAC_Mineral_Development...
  --> Success: Found 3 graphite records.
Processing INDOPAC_Mineral_Exploration...
  --> Success: Found 42 graphite records.


,USGS Facility ID,Country (Short Form),Facility Name,Facility Type,Commodity Type,Commodity,Commodity Product,Multiple Commodities,Multiple Products,Year,...,OwnerName3,OwnerName4,OwnerName5,Owner Type,Data Source,Location Source 2,InfSource8,InfSource9,Exploration ID,Exploration Status
0,MYS284,Malaysia,Plant in Kawasan Perindustrian,Mineral Processing Plants,Graphite,Synthetic Graphite,Carbon electrodes,N,N,-999.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LKA011,Sri Lanka,Kahatagaha Mine,Mines and Quarries,Graphite,None,None,N,N,780.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LKA017,Sri Lanka,Ragedara Mine,Mines and Quarries,Graphite,None,None,N,N,500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LKA018,Sri Lanka,Bogala Mine,Mines and Quarries,Graphite,None,None,N,N,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VNM097,Vietnam,Nam Thi Mine,Mines and Quarries,Graphite,None,None,N,N,-999.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,VNM199,Vietnam,Graphite Purification Plant,Mineral Processing Plants,Graphite,None,None,N,N,70.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,Bhutan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,2021.0,https://openjicareport.jica.go.jp/pdf/12357372...,http://www.seanpatricklong.com/uploads/1/4/5/6...,None,http://www.citypopulation.de/en/bhutan/admin/,NaN,NaN
7,NaN,Bhutan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,2021.0,https://openjicareport.jica.go.jp/pdf/12357372...,None,None,http://www.citypopulation.de/en/bhutan/admin/,NaN,NaN
8,NaN,Sri Lanka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,2022.0,https://www.ceylongraphite.com/work-sites/m1-d...,https://www.ceylongraphite.com/wp-content/uplo...,None,None,NaN,NaN
9,NaN,Bhutan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,NaN,2021.0,https://openjicareport.jica.go.jp/pdf/12357372...,https://repository.unescap.org/bitstream/handl...,None,None,BTx002,Inactive


## 2.4. SW Asia

In [38]:
# Path to SW Asia GDB
swa_path = os.path.join(base_dir, "Compilation of GIS SW Asia", "SWAsia_GIS.gdb.zip")
# Inspect
layers = inspect_gdb(swa_path)

--- Layers in GDB ---
                               name    geometry_type
0          SWA_Infra_Power_Stations          Point Z
1        SWA_Infra_OG_LNG_Terminals          Point Z
2      SWA_OG_Resources_Recoverable     MultiPolygon
3     SWA_OG_Provinces_Conventional     MultiPolygon
4       SWA_OG_Provinces_Continuous     MultiPolygon
5      SWA_Mineral_Resources_Potash     MultiPolygon
6   SWA_Mineral_Resources_Phosphate     MultiPolygon
7      SWA_Mineral_Resources_Copper     MultiPolygon
8        SWA_Mineral_Resources_Coal     MultiPolygon
9         SWA_Infra_Transport_Ports            Point
10     SWA_Infra_Power_Transmission  MultiLineString
11             SWA_Mineral_Deposits            Point
12          SWA_Mineral_Exploration            Point
13            SWA_Mineral_Facilties            Point


c:\Users\157412\.conda\envs\pemmss\Lib\site-packages\pyogrio\core.py:130: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Point' is converted to 'Point Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))


In [39]:
# Note the intentional typo in 'SWA_Mineral_Facilties' to match the database layer name
SWA_ALIASES = {
    'SWA_Mineral_Facilties': {
        'FeatureUID': 'USGS Facility ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'LocConfid': 'Location Accuracy',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'MemoLoc': 'Location Description',
        'FeatureTyp': 'Facility Type',
        'FeatureNam': 'Facility Name',
        'Label1': 'Label',
        'DsgAttr01': 'Commodity Type',
        'DsgAttr02': 'Commodity',
        'DsgAttr03': 'Commodity Product',
        'DsgAttr04': 'Multiple Commodities',
        'DsgAttr05': 'Multiple Products',
        'DsgAttr06': 'Year',
        'DsgAttr07': 'Annual Production Capacity',
        'DsgAttr08': 'Capacity Units',
        'DsgAttr09': 'Capacity Notes',
        'DsgAttr10': 'Shared Capacity',
        'LocOpStat': 'Facility Status',
        'MemoOther': 'Facility Comments',
        'OperateNam': 'Major Operating Company',
        'OwnerName1': 'Equity Owner 1',
        'OwnerName2': 'Equity Owner 2',
        'OwnerName3': 'Equity Owner 3',
        'OwnerName4': 'Equity Owner 4',
        'OwnerName5': 'Equity Owner 5',
        'InfSource1': 'Mineral Facility Data Source',
        'LocSource1': 'GIS Data Source'
    },
    'SWA_Mineral_Deposits': {
        'FeatureUID': 'Deposit ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Deposit Name',
        'NameVar': 'Alternate Name(s)',
        'Label1': 'Label',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Commodity 8',
        'DsgAttr09': 'Deposit Type',
        'DsgAttr10': 'USGS MRData ID',
        'InfSource1': 'Data Source',
        'InfSource2': 'Location Source 2'
    },
    'SWA_Mineral_Exploration': {
        'FeatureUID': 'Exploration ID',
        'Latitude': 'DD Latitude',
        'Longitude': 'DD Longitude',
        'Country': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'FeatureNam': 'Site Name',
        'NameVar': 'Alternate Name(s)',
        'Label1': 'Label',
        'MemoLoc': 'Location Notes',
        'LocOpStat': 'Exploration Status',
        'DsgAttr01': 'Commodity 1',
        'DsgAttr02': 'Commodity 2',
        'DsgAttr03': 'Commodity 3',
        'DsgAttr04': 'Commodity 4',
        'DsgAttr05': 'Commodity 5',
        'DsgAttr06': 'Commodity 6',
        'DsgAttr07': 'Commodity 7',
        'DsgAttr08': 'Commodity 8',
        'OwnerName': 'Owner Name',
        'DsgAttr09': 'Owner Type',
        'DsgAttr10': 'Owner Share',
        'DsgAttr11': 'Project Active',
        'DsgAttr12': 'USGS WMED ID',
        'InfSource1': 'Data Source',
        'InfSource2': 'Location Source 2'
    }
}

In [40]:
def load_swa_data_and_aliases(gdb_path):
    all_graphite = []
    
    for layer_name, mapping in SWA_ALIASES.items():
        try:
            print(f"Processing {layer_name}...")
            # Using pyogrio to efficiently load the specific layer
            gdf = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio")
            
            # Apply the aliases
            gdf = gdf.rename(columns=mapping)
            
            # Perform the case-insensitive search across all text columns
            mask = gdf.astype(str).apply(
                lambda x: x.str.contains('graphite', case=False, na=False)
            ).any(axis=1)
            
            hits = gdf[mask].copy()
            
            if not hits.empty:
                hits['Source_Layer'] = layer_name
                hits['Region'] = 'SW Asia'
                all_graphite.append(hits)
                print(f"  --> Success: Found {len(hits)} graphite records.")
            else:
                print(f"  --> No graphite found.")
                
        except Exception as e:
            print(f"  --> Error loading {layer_name}: {e}")

    if all_graphite:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            return pd.concat(all_graphite, ignore_index=True)
    return None

swa_master = load_swa_data_and_aliases(swa_path)

if swa_master is not None:
    display(swa_master.head(10))

Processing SWA_Mineral_Facilties...
  --> Success: Found 8 graphite records.
Processing SWA_Mineral_Deposits...
  --> Success: Found 20 graphite records.
Processing SWA_Mineral_Exploration...
  --> Success: Found 4 graphite records.


,USGS Facility ID,Label,Country (Short Form),Facility Name,Facility Type,Commodity Type,Commodity,Commodity Product,Multiple Commodities,Multiple Products,...,USGS MRData ID,DsgAttr11,DsgAttr12,DsgAttr13,Data Source,Exploration ID,Site Name,Exploration Status,Owner Type,Location Notes
0,IND298,IND298,India,Mine in Jharkhand Province,Mines and Quarries,Metal,Graphite,None,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IND659,None,India,Quarry in Tamil Nadu Province,Mines and Quarries,Metal,Graphite,None,Y,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IND758,IND758,India,Tamil Nadu Minerals Graphite Benefication Plant,Mines and Quarries,Metal,Graphite,None,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,PRK017,PRK17,"Korea, North",Sinwon Mine,Mines and Quarries,Industrial,Graphite,None,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,PRK035,PRK35,"Korea, North",Wonri Mine,Mines and Quarries,Industrial,Graphite,Graphite (amorphous),N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,PRK041,PRK41,"Korea, North",Yongchon (Ryongchon) Mine,Mines and Quarries,Industrial,Graphite,None,N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,PRK081,PRK81,"Korea, North",Heungsan Mine,Mines and Quarries,Industrial,Graphite,Graphite (crystalline flake),N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,PRK092,PRK92,"Korea, North",Jeongchon Mine,Mines and Quarries,Industrial,Graphite,Graphite (crystalline flake),N,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,AFd28,Afghanistan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,Unknown graphite,Yes,"Schulz and Briskey, 2005 (USGS Open-File Repor...",NaN,NaN,NaN,NaN,NaN
9,NaN,AFd52,Afghanistan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,Unknown graphite,Yes,"Schulz and Briskey, 2005 (USGS Open-File Repor...",NaN,NaN,NaN,NaN,NaN


## 2.5. Latin American and Caribbean

In [41]:
# Path to LAC folder
lac_path = os.path.join(base_dir, "Compilation of GIS Latin American and Carib")


In [42]:
# The column headers in the LAC CSV/Excel files are different from the GDBs.
# We map them to match the aliases used in Africa, China, and Indopacific.
LAC_ALIASES = {
    'MINFAC_LAC': {
        'FACID': 'USGS Facility ID',
        'DDLAT': 'DD Latitude',
        'DDLONG': 'DD Longitude',
        'COUNTRY': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'LOCNAME': 'Facility Name',
        'LOCDESC': 'Location Description',
        'COMMODITY': 'Commodity',
        'MIN_PROD': 'Commodity Product',
        'FACTYPE': 'Facility Type',
        'OPERATOR': 'Major Operating Company',
        'OWNER': 'Equity Owner 1',
        'ANNCAP': 'Annual Production Capacity',
        'UNITS': 'Capacity Units',
        'STATUS': 'Facility Status',
        'LOCACC': 'Location Accuracy',
        'COMMENTS': 'Facility Comments'
    },
    'EXPLORE_LAC': {
        'FACID': 'Exploration ID',
        'DDLAT': 'DD Latitude',
        'DDLONG': 'DD Longitude',
        'COUNTRY': 'Country (Short Form)',
        'ADM1': 'ADM1 Name',
        'PROJNAME': 'Site Name',
        'COMM': 'Commodity 1', # In LAC, commodities are often comma-separated in one column
        'OPERATOR': 'Owner Name',
        'OWNER': 'Equity Owner 1',
        'LOCACC': 'Location Accuracy',
        'PROJTYPE': 'Exploration Status'
    }
}


In [43]:
def load_lac_data_and_aliases(folder_path):
    all_graphite = []
    
    if not os.path.exists(folder_path):
        print(f"Error: Could not find folder at {folder_path}")
        return None

    files = [f for f in os.listdir(folder_path) if f.endswith(('.xlsx', '.xls', '.csv'))]
    
    for file_name in files:
        layer_key = next((key for key in LAC_ALIASES.keys() if key in file_name), None)
        
        if not layer_key:
            continue
            
        try:
            print(f"Processing {file_name}...")
            
            # Load CSV or Excel (Added encoding='latin1' to handle special characters)
            file_path = os.path.join(folder_path, file_name)
            if file_name.endswith('.csv'):
                df = pd.read_csv(file_path, encoding='latin1')
            else:
                df = pd.read_excel(file_path)
            
            # Apply the aliases
            df = df.rename(columns=LAC_ALIASES[layer_key])
            
            # Case-insensitive search across all text columns
            text_cols = df.select_dtypes(include=['object']).columns
            mask = df[text_cols].astype(str).apply(
                lambda x: x.str.contains('graphite', case=False, na=False)
            ).any(axis=1)
            
            hits = df[mask].copy()
            
            if not hits.empty:
                # Convert to GeoDataFrame
                if 'DD Latitude' in hits.columns and 'DD Longitude' in hits.columns:
                    hits = hits.dropna(subset=['DD Latitude', 'DD Longitude'])
                    
                    hits = gpd.GeoDataFrame(
                        hits, 
                        geometry=gpd.points_from_xy(hits['DD Longitude'], hits['DD Latitude']),
                        crs="EPSG:4326" 
                    )
                
                hits['Source_Layer'] = layer_key
                hits['Region'] = 'Latin America & Caribbean'
                all_graphite.append(hits)
                print(f"  --> Success: Found {len(hits)} graphite records.")
            else:
                print(f"  --> No graphite found.")
                
        except Exception as e:
            print(f"  --> Error loading {file_name}: {e}")

    if all_graphite:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", FutureWarning)
            return pd.concat(all_graphite, ignore_index=True)
    return None

lac_master = load_lac_data_and_aliases(lac_path)

if lac_master is not None:
    display(lac_master.head(10))

Processing EXPLORE_LAC.csv...
  --> No graphite found.
Processing MINFAC_LAC.csv...
  --> Success: Found 8 graphite records.


,ROWID,ISO_CC,FACID_NUM,USGS Facility ID,MULTI_COMM,MULTI_PROD,Country (Short Form),YEAR,COMMTYPE,Commodity,...,COMBOCAP,Facility Status,Location Accuracy,DD Latitude,DD Longitude,SOURCEID,Facility Comments,geometry,Source_Layer,Region
0,349,BR,100,BR100,NaN,NaN,Brazil,2014,Industrial,Graphite,...,***BR100-102,A,E,-20.4412,-45.1334,351,NaN,POINT (-45.1334 -20.4412),MINFAC_LAC,Latin America & Caribbean
1,358,BR,109,BR109,NaN,NaN,Brazil,2014,Industrial,Graphite,...,NaN,A,E,-15.7794,-40.3215,361,NaN,POINT (-40.3215 -15.7794),MINFAC_LAC,Latin America & Caribbean
2,368,BR,113,BR113,NaN,NaN,Brazil,2014,Industrial,Graphite,...,NaN,A,E,-20.0379,-44.4351,368,NaN,POINT (-44.4351 -20.0379),MINFAC_LAC,Latin America & Caribbean
3,401,BR,101,BR101,NaN,NaN,Brazil,2014,Industrial,Graphite,...,***BR100-102,A,E,-19.8734,-44.0582,352,NaN,POINT (-44.0582 -19.8734),MINFAC_LAC,Latin America & Caribbean
4,440,BR,102,BR102,NaN,NaN,Brazil,2014,Industrial,Graphite,...,***BR100-102,A,E,-16.2026,-39.9472,353,NaN,POINT (-39.9472 -16.2026),MINFAC_LAC,Latin America & Caribbean
5,1145,MX,84,MX84,NaN,NaN,Mexico,2014,Industrial,Graphite,...,***MX84-86,A,A,28.4835,-110.6839,1187,NaN,POINT (-110.6839 28.4835),MINFAC_LAC,Latin America & Caribbean
6,1217,MX,85,MX85,NaN,NaN,Mexico,2014,Industrial,Graphite,...,***MX84-86,A,A,27.4580,-108.9363,1188,NaN,POINT (-108.9363 27.458),MINFAC_LAC,Latin America & Caribbean
7,1253,MX,86,MX86,NaN,NaN,Mexico,2014,Industrial,Graphite,...,***MX84-86,A,A,27.4580,-108.9363,1189,NaN,POINT (-108.9363 27.458),MINFAC_LAC,Latin America & Caribbean


# 3. Setting up output dataframes

In [45]:
# --- FINAL MERGE AND EXPORT CELL ---

# 1. Combine all regional GeoDataFrames
master_datasets = [africa_master, china_master, indopac_master, swa_master, lac_master]

# Filter out any regions that might have returned None (if no graphite was found)
valid_datasets = [df for df in master_datasets if df is not None]

# Concatenate into one massive global spatial dataset
global_graphite_gdf = pd.concat(valid_datasets, ignore_index=True)

print(f"\n--- GLOBAL MERGE COMPLETE ---")
print(f"Total Graphite Records Identified: {len(global_graphite_gdf)}")

# 2. Reorder columns to put the most important ones first for easier reading
front_cols = [
    'Region', 'Source_Layer', 'Country (Short Form)', 'Facility Name', 'Site Name', 'Deposit Name',
    'Commodity Type', 'Commodity', 'Commodity 1', 'Facility Status', 'Exploration Status',
    'Annual Production Capacity', 'Capacity Units', 'Major Operating Company', 'Owner Name'
]

# Get remaining columns that exist in the dataframe
remaining_cols = [c for c in global_graphite_gdf.columns if c not in front_cols and c != 'geometry']

# Ensure we only try to sort columns that actually exist in the merged dataset
front_cols_existing = [c for c in front_cols if c in global_graphite_gdf.columns]

# Reapply column order, always keeping geometry at the end for spatial formats
final_cols = front_cols_existing + remaining_cols + ['geometry']
global_graphite_gdf = global_graphite_gdf[final_cols]

# 3. Export to CSV for Excel/Tabular Analysis
csv_export_path = os.path.join(base_dir, "Global projects - Clean data", "Global_Graphite_Projects.csv")
# Drop the geometry column for the pure CSV export so it opens cleanly in Excel
global_graphite_gdf.drop(columns='geometry').to_csv(csv_export_path, index=False)
print(f"Saved Tabular Data to: {csv_export_path}")

# 4. Export to a Spatial Format (Shapefile or GeoJSON) for QGIS/ArcGIS
geojson_export_path = os.path.join(base_dir, "Global projects - Clean data", "Global_Graphite_Projects.geojson")
# GeoJSON is generally safer and less restrictive with column names/lengths than Shapefiles
global_graphite_gdf.to_file(geojson_export_path, driver='GeoJSON')
print(f"Saved Spatial Data to: {geojson_export_path}")

# Quick preview
display(global_graphite_gdf.head())


--- GLOBAL MERGE COMPLETE ---
Total Graphite Records Identified: 184
Saved Tabular Data to: C:\Users\157412\OneDrive - UTS (1)\ISF Projects - 23204_Quadrature_Mineral Availability for Renewables PEMMS\3. Resources\Graphite Production\Sciencebase data\Global projects - Clean data\Global_Graphite_Projects.csv
Saved Spatial Data to: C:\Users\157412\OneDrive - UTS (1)\ISF Projects - 23204_Quadrature_Mineral Availability for Renewables PEMMS\3. Resources\Graphite Production\Sciencebase data\Global projects - Clean data\Global_Graphite_Projects.geojson


C:\Users\157412\AppData\Local\Temp\ipykernel_38372\2813888070.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  global_graphite_gdf = pd.concat(valid_datasets, ignore_index=True)


,Region,Source_Layer,Country (Short Form),Facility Name,Site Name,Deposit Name,Commodity Type,Commodity,Commodity 1,Facility Status,...,ISO_CC,FACID_NUM,MULTI_COMM,MULTI_PROD,YEAR,COMMTYPE,ECAP,COMBOCAP,SOURCEID,geometry
0,NaN,AFR_Mineral_Facilities,Madagascar,Gallois Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT Z (48.94507 -19.2302 0)
1,NaN,AFR_Mineral_Facilities,Madagascar,Graphmada Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT Z (48.99125 -18.91881 0)
2,NaN,AFR_Mineral_Facilities,Madagascar,Mine in Atsinanana Region,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT Z (49.21805 -18.44545 0)
3,NaN,AFR_Mineral_Facilities,Madagascar,Sahamamy-Sahasoa Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT Z (48.9757 -18.56834 0)
4,NaN,AFR_Mineral_Facilities,Mozambique,Ancuabe Graphite Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT Z (40.00067 -13.00407 0)


In [46]:
global_graphite_gdf.to_csv('outputs/Global_Graphite_Projects.csv', index=False)